In [1]:
import os
import cv2

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset"
target_path = r"/mnt/lustre/lalit47/down/seg/Spinal Cord"

# Source folders
train_images_folder = os.path.join(source_path, "train_images")
train_masks_folder = os.path.join(source_path, "train_masks")

val_images_folder = os.path.join(source_path, "val_images")
val_masks_folder = os.path.join(source_path, "val_masks")

test_images_folder = os.path.join(source_path, "test_images")
test_masks_folder = os.path.join(source_path, "test_masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


def process_and_save(src, dst, is_mask=False):
    """
    Read PNG in grayscale,
    resize to 256x256,
    save as PNG.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    if is_mask:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_NEAREST)
    else:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)
    return True


def get_png_stems(folder):
    """
    Get PNG file names without extension.
    """

    stems = []

    for file in os.listdir(folder):
        if file.lower().endswith(".png"):
            stem = os.path.splitext(file)[0]
            stems.append(stem)

    stems.sort()
    return stems


def process_image_mask_pairs(image_folder, mask_folder, image_output, mask_output):
    """
    Process all PNG image-mask pairs from given folders.
    Image and mask must have the same filename.
    """

    image_stems = get_png_stems(image_folder)
    mask_stems = get_png_stems(mask_folder)

    common_stems = sorted(list(set(image_stems) & set(mask_stems)))

    missing_masks = sorted(list(set(image_stems) - set(mask_stems)))
    missing_images = sorted(list(set(mask_stems) - set(image_stems)))

    if missing_masks:
        print(f"Images without masks in {image_folder}: {len(missing_masks)}")

    if missing_images:
        print(f"Masks without images in {mask_folder}: {len(missing_images)}")

    print(f"Processing {len(common_stems)} pairs from {image_folder}")

    for stem in common_stems:
        image_src = os.path.join(image_folder, stem + ".png")
        mask_src = os.path.join(mask_folder, stem + ".png")

        image_dst = os.path.join(image_output, stem + ".png")
        mask_dst = os.path.join(mask_output, stem + ".png")

        process_and_save(image_src, image_dst, is_mask=False)
        process_and_save(mask_src, mask_dst, is_mask=True)


# =========================
# Main processing
# =========================

process_image_mask_pairs(
    train_images_folder,
    train_masks_folder,
    train_images_out,
    train_masks_out
)

process_image_mask_pairs(
    val_images_folder,
    val_masks_folder,
    val_images_out,
    val_masks_out
)

process_image_mask_pairs(
    test_images_folder,
    test_masks_folder,
    test_images_out,
    test_masks_out
)

print("\nDataset preprocessing completed successfully!")

Processing 8668 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/train_images
Processing 895 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/val_images
Processing 660 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/test_images

Dataset preprocessing completed successfully!


In [2]:
import os
import cv2
import numpy as np

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset"
target_path = r"/mnt/lustre/lalit47/down/seg/Spinal Cord"

# Source folders
train_images_folder = os.path.join(source_path, "train_images")
train_masks_folder = os.path.join(source_path, "train_masks")

val_images_folder = os.path.join(source_path, "val_images")
val_masks_folder = os.path.join(source_path, "val_masks")

test_images_folder = os.path.join(source_path, "test_images")
test_masks_folder = os.path.join(source_path, "test_masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


# =========================
# MASK LABEL MAPPING
# =========================
# Original grayscale values -> class IDs 0 to 9
MASK_LABEL_MAP = {
    0: 0,
    14: 1,
    19: 2,
    38: 3,
    52: 4,
    57: 5,
    75: 6,
    89: 7,
    113: 8,
    128: 9,
}

IGNORE_VALUE = 255


def remap_mask(mask):
    """
    Convert grayscale mask values to class indices.

    Input values:
        [0, 14, 19, 38, 52, 57, 75, 89, 113, 128]

    Output values:
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    """

    remapped = np.full(mask.shape, IGNORE_VALUE, dtype=np.uint8)

    for old_value, new_value in MASK_LABEL_MAP.items():
        remapped[mask == old_value] = new_value

    unknown_values = sorted(set(np.unique(mask).tolist()) - set(MASK_LABEL_MAP.keys()))

    if unknown_values:
        print(f"Warning: unknown mask values found: {unknown_values}")

    return remapped


def process_and_save(src, dst, is_mask=False):
    """
    Read PNG in grayscale,
    resize to 256x256,
    map mask values if mask,
    save as PNG.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    if is_mask:
        # Resize mask with nearest-neighbor to preserve label values
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_NEAREST)

        # Convert grayscale label values to class IDs 0-9
        im = remap_mask(im)

        # Final safety check
        unique_values = np.unique(im)
        invalid_values = unique_values[(unique_values != IGNORE_VALUE) & (unique_values >= 10)]

        if len(invalid_values) > 0:
            raise ValueError(
                f"Invalid mapped labels in {src}: {invalid_values}. "
                f"Expected labels 0-9 or {IGNORE_VALUE}."
            )

    else:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)
    return True


def get_png_stems(folder):
    """
    Get PNG file names without extension.
    """

    stems = []

    for file in os.listdir(folder):
        if file.lower().endswith(".png"):
            stem = os.path.splitext(file)[0]
            stems.append(stem)

    stems.sort()
    return stems


def process_image_mask_pairs(image_folder, mask_folder, image_output, mask_output):
    """
    Process all PNG image-mask pairs from given folders.
    Image and mask must have the same filename.
    """

    image_stems = get_png_stems(image_folder)
    mask_stems = get_png_stems(mask_folder)

    common_stems = sorted(list(set(image_stems) & set(mask_stems)))

    missing_masks = sorted(list(set(image_stems) - set(mask_stems)))
    missing_images = sorted(list(set(mask_stems) - set(image_stems)))

    if missing_masks:
        print(f"Images without masks in {image_folder}: {len(missing_masks)}")

    if missing_images:
        print(f"Masks without images in {mask_folder}: {len(missing_images)}")

    print(f"Processing {len(common_stems)} pairs from {image_folder}")

    for idx, stem in enumerate(common_stems):
        image_src = os.path.join(image_folder, stem + ".png")
        mask_src = os.path.join(mask_folder, stem + ".png")

        image_dst = os.path.join(image_output, stem + ".png")
        mask_dst = os.path.join(mask_output, stem + ".png")

        ok_img = process_and_save(image_src, image_dst, is_mask=False)
        ok_mask = process_and_save(mask_src, mask_dst, is_mask=True)

        if ok_img and ok_mask and idx < 5:
            saved_mask = cv2.imread(mask_dst, cv2.IMREAD_GRAYSCALE)
            print(f"{stem}.png saved mask unique: {np.unique(saved_mask)}")


# =========================
# Main processing
# =========================

process_image_mask_pairs(
    train_images_folder,
    train_masks_folder,
    train_images_out,
    train_masks_out,
)

process_image_mask_pairs(
    val_images_folder,
    val_masks_folder,
    val_images_out,
    val_masks_out,
)

process_image_mask_pairs(
    test_images_folder,
    test_masks_folder,
    test_images_out,
    test_masks_out,
)

print("\nDataset preprocessing completed successfully!")

Processing 8668 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/train_images
predict11_scaled-A0067_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0069_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame10.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame100.png saved mask unique: [1 3 4 5 6 8]
Processing 895 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/val_images
predict10_scaled-A0091_frame1.png saved mask unique: [1 3 4 6 8 9]
predict10_scaled-A0091_frame10.png saved mask unique: [1 3 4 6 8 9]
predict10_scaled-A0091_frame100.png saved mask unique: [1 3 4 5 6 8]
predict10_scaled-A0091_frame101.png saved mask unique: [1 3 4 5 6 8]
predict10_scaled-A0091_frame102.png saved mask unique: [1 3 4 5 6 8]
Processing 660 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury Segmentat

In [3]:
import os
import cv2
import numpy as np

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset"
target_path = r"/mnt/lustre/lalit47/down/seg/Spinal Cord"

# Source folders
train_images_folder = os.path.join(source_path, "train_images")
train_masks_folder = os.path.join(source_path, "train_masks")

val_images_folder = os.path.join(source_path, "val_images")
val_masks_folder = os.path.join(source_path, "val_masks")

test_images_folder = os.path.join(source_path, "test_images")
test_masks_folder = os.path.join(source_path, "test_masks")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


# =========================
# MASK LABEL MAPPING
# =========================

MASK_LABEL_MAP = {
    0: 0,
    14: 1,
    19: 2,
    38: 3,
    52: 4,
    57: 5,
    75: 6,
    89: 7,
    113: 8,
    128: 9,
}

VALID_MASK_VALUES = set(MASK_LABEL_MAP.keys())


def process_image(src):
    image = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print(f"Failed to read image: {src}")
        return None

    image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_LINEAR)
    return image


def process_mask(src):
    mask = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"Failed to read mask: {src}")
        return None

    mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

    unique_values = set(np.unique(mask).tolist())
    unknown_values = sorted(unique_values - VALID_MASK_VALUES)

    if unknown_values:
        print(f"Skipping sample because unknown mask values found in {src}: {unknown_values}")
        return None

    remapped = np.zeros(mask.shape, dtype=np.uint8)

    for old_value, new_value in MASK_LABEL_MAP.items():
        remapped[mask == old_value] = new_value

    return remapped


def get_png_stems(folder):
    stems = []

    for file in os.listdir(folder):
        if file.lower().endswith(".png"):
            stems.append(os.path.splitext(file)[0])

    stems.sort()
    return stems


def process_image_mask_pairs(image_folder, mask_folder, image_output, mask_output):
    image_stems = get_png_stems(image_folder)
    mask_stems = get_png_stems(mask_folder)

    common_stems = sorted(list(set(image_stems) & set(mask_stems)))

    missing_masks = sorted(list(set(image_stems) - set(mask_stems)))
    missing_images = sorted(list(set(mask_stems) - set(image_stems)))

    if missing_masks:
        print(f"Images without masks in {image_folder}: {len(missing_masks)}")

    if missing_images:
        print(f"Masks without images in {mask_folder}: {len(missing_images)}")

    print(f"\nProcessing {len(common_stems)} pairs from {image_folder}")

    copied_count = 0
    skipped_count = 0

    for stem in common_stems:
        image_src = os.path.join(image_folder, stem + ".png")
        mask_src = os.path.join(mask_folder, stem + ".png")

        image = process_image(image_src)
        mask = process_mask(mask_src)

        # If either image or mask failed, skip the whole sample
        if image is None or mask is None:
            skipped_count += 1
            continue

        image_dst = os.path.join(image_output, stem + ".png")
        mask_dst = os.path.join(mask_output, stem + ".png")

        cv2.imwrite(image_dst, image)
        cv2.imwrite(mask_dst, mask)

        copied_count += 1

        if copied_count <= 5:
            print(f"{stem}.png saved mask unique: {np.unique(mask)}")

    print(f"Copied samples: {copied_count}")
    print(f"Skipped samples: {skipped_count}")


# =========================
# Main processing
# =========================

process_image_mask_pairs(
    train_images_folder,
    train_masks_folder,
    train_images_out,
    train_masks_out,
)

process_image_mask_pairs(
    val_images_folder,
    val_masks_folder,
    val_images_out,
    val_masks_out,
)

process_image_mask_pairs(
    test_images_folder,
    test_masks_folder,
    test_masks_out.replace("masks", "images"),
    test_masks_out,
)

print("\nDataset preprocessing completed successfully!")


Processing 8668 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/train_images
predict11_scaled-A0067_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0069_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame1.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame10.png saved mask unique: [1 3 4 5 6 8]
predict11_scaled-A0070_frame100.png saved mask unique: [1 3 4 5 6 8]
Skipping sample because unknown mask values found in /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/train_masks/predict4_scaled-A0130_frame47.png: [76, 77, 83, 87]
Copied samples: 8667
Skipped samples: 1

Processing 895 pairs from /mnt/lustre/lalit47/us_data/new_data/Spinal cord injury SegmentationDataset/val_images
predict10_scaled-A0091_frame1.png saved mask unique: [1 3 4 6 8 9]
predict10_scaled-A0091_frame10.png saved mask unique: [1 3 4 6 8 9]
predict10_scaled-A0091_frame100.png saved mask unique: [1 3 4 5

In [4]:
import os
import cv2
import numpy as np

target_path = r"/mnt/lustre/lalit47/down/seg/Spinal Cord"

for split in ["train", "val", "test"]:
    mask_dir = os.path.join(target_path, split, "masks")
    all_values = set()

    for file in os.listdir(mask_dir):
        if not file.lower().endswith(".png"):
            continue

        mask = cv2.imread(os.path.join(mask_dir, file), cv2.IMREAD_GRAYSCALE)
        vals = np.unique(mask).tolist()
        all_values.update(vals)

        bad = [v for v in vals if v > 9]
        if bad:
            print(f"BAD MASK: {split}/{file}, values={vals}")

    print(split, "all mask values:", sorted(all_values))

train all mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
val all mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
test all mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
